In [ ]:
import pandas as pd

In [2]:
cache = '../data/processed/retail_raw_combined.parquet'
df = pd.read_parquet(cache)

In [3]:
df.head(3)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom


## Product Dimension | Main_Category + Subcategory

Turn the `Description` column into a structured **product dimension** (one row per `StockCode`)
with `Main_Category` + `Subcategory`.

Pipeline:
1. Unique `StockCode` + most frequent description (~5k products, not 1M+ rows)
2. **Rules pass**, keywords from `../data/product_taxonomy.json`, first match wins,
   auditable `Classification_Method`. Themes/colours are NOT categories (a Christmas mug
   stays Kitchen & Dining).
3. **LLM for the residue, one product per request**, batching made the 3B model anchor on
   its first answer and copy it down the batch, so each item is classified in its own call

> **Editing the taxonomy:** `../data/product_taxonomy.json` is the single source of truth,
> `categories` is an *ordered array* (order = classification order, first match wins; a
> category may appear more than once: specific rules early, generic catch-alls last).
> Each subcategory has a `description` (feeds the LLM prompt) and optional `rules`
> keywords.


In [ ]:
import json
import re
import time
import os
import requests as rq
from concurrent.futures import ThreadPoolExecutor, as_completed

OLLAMA_URL = "http://127.0.0.1:11434/api/chat"
MODEL = "qwen2.5:3b"
TEST_MODE = False
TEST_ITEMS = 30        # items to classify in TEST_MODE

MAX_WORKERS = 2        # parallel requests to the local Ollama server
FLUSH_EVERY = 25       # checkpoint rows buffered before writing to disk

TAXONOMY_PATH = "../data/product_taxonomy.json"
CKPT_PATH = "../data/processed/product_categories_checkpoint.csv"
PRODUCT_DIM_PATH = "../data/processed/product_dimension.csv"
REVIEW_PATH = "../data/processed/needs_review.csv"


## Unique product table

One row per `StockCode`, the most frequent description wins. Description variants of the
same code collapse into a single product row on the join back.


In [5]:
products = (
    df[["StockCode", "Description"]]
    .dropna(subset=["Description"])
    .assign(Description=lambda d: d["Description"].str.strip().str.upper())
    .groupby("StockCode", as_index=False)
    .agg(Description=("Description", lambda s: s.mode().iat[0]))
)
products["StockCode"] = products["StockCode"].astype(str)
print(f"{len(products)} unique products (one description per StockCode)")
products.head()


4950 unique products (one description per StockCode)


,StockCode,Description
0,10002,INFLATABLE POLITICAL GLOBE
1,10002R,ROBOT PENCIL SHARPNER
2,10080,GROOVY CACTUS INFLATABLE
3,10109,BENDY COLOUR PENCILS
4,10120,DOGGY RUBBER


## Rules pass

### Load Taxonomy Setup
Keywords live in `../data/product_taxonomy.json`. We load the JSON and merge repeated category blocks to preserve the hierarchy.

In [ ]:
with open(TAXONOMY_PATH) as f:
    TAXONOMY_BLOCKS = json.load(f)["categories"]

# merge repeated category blocks, preserve order
TAXONOMY = {}
for block in TAXONOMY_BLOCKS:
    cat = TAXONOMY.setdefault(
        block["name"], {"description": block["description"], "subcategories": {}}
    )
    for s in block["subcategories"]:
        cat["subcategories"][s["name"]] = s["description"]

### Rule Compilation
We compile rules with a `kw()` helper that tolerates plurals and space/hyphen variants. Non-Product rows are mapped for **exact** matching by StockCode or description.

In [ ]:
def kw(*words):
    parts = [w.replace(" ", r"[\s-]?") for w in words]
    return re.compile(r"\b(?:" + "|".join(parts) + r")(?:E?S)?\b")

RULES = []
EXACT = {}
for block in TAXONOMY_BLOCKS:
    for s in block["subcategories"]:
        for r in s.get("rules", []):
            RULES.append((block["name"], s["name"], kw(r), r))
        for code in s.get("exact_codes", []):
            EXACT[code] = (block["name"], s["name"])
        for d in s.get("exact_descriptions", []):
            EXACT[d] = (block["name"], s["name"])

for main, sub, _, _ in RULES:
    assert sub in TAXONOMY[main]["subcategories"], f"rule target missing: {main}/{sub}"

### Classification Logic
Exact matches run first to prevent keyword regex misfires (e.g., `COFFEE` contains `FEE`, `POSTCARD` contains `POST`). If no exact match is found, we fall back to the compiled regex rules.

In [ ]:
def classify_rule(code, desc):
    text = re.sub(r"\s+", " ", str(desc)).upper().strip()
    key = text.rstrip(".")
    if code in EXACT:
        return (*EXACT[code], f"exact:{code}")
    if key in EXACT:
        return (*EXACT[key], f"exact:{key}")
    for main, sub, pat, r in RULES:
        if pat.search(text):
            return main, sub, f"rule:{r}"
    return None

### Apply Rules and Evaluate
Apply the classification engine to the dataset. Anything left unmatched is set aside for the LLM pipeline.

In [ ]:
classified = products.apply(lambda r: classify_rule(r["StockCode"], r["Description"]), axis=1)
products["Main_Category"] = classified.map(lambda t: t[0] if t else None)
products["Subcategory"] = classified.map(lambda t: t[1] if t else None)
products["Confidence"] = classified.map(lambda t: "high" if t else None)
products["Classification_Method"] = classified.map(lambda t: t[2] if t else None)

to_classify = products[products["Main_Category"].isna()].copy()
n = len(products) - len(to_classify)
print(f"rules matched {n}/{len(products)} ({n / len(products) * 100:.1f}%), "
      f"{len(to_classify)} left for the LLM")
print(products["Main_Category"].value_counts(dropna=False).to_string())

## Taxonomy prompt and JSON schema

The prompt is generated from the taxonomy file, every main category and subcategory
description becomes a classification rule, plus a few examples. The `format` field makes
Ollama enforce the JSON schema (structured outputs).

In [ ]:
taxonomy_text = "\n".join(
    f"- {main}: {info['description']}\n"
    + "\n".join(f"    * {sub}: {desc}" for sub, desc in info["subcategories"].items())
    for main, info in TAXONOMY.items()
)

In [ ]:
SYSTEM_PROMPT = f"""You classify products from a UK online retailer selling all-occasion giftware, using only the short product name.

Taxonomy (Main_Category: description, then its Subcategories with descriptions):
{taxonomy_text}

Choose exactly one Main_Category and one Subcategory from this taxonomy for each product.

Rules:
- Classify by the product's primary purpose, not its colour or pattern. "JUMBO BAG RED RETROSPOT" is a bag; "PINK CHERRY LIGHTS" is lighting.
- A functional item (mug, tea towel, candle, apron) with a seasonal print stays in its function category. Seasonal wording (CHRISTMAS, EASTER) is NOT a category.
- Use "Non-Product" only for postage, fees, adjustments, damages or samples. Most of those are already handled before you.
- If nothing fits, use "Other" / "Unclassified". Never invent categories or subcategories.
- Confidence: "high" if the name clearly names the product type, "medium" if you inferred it, "low" if very unsure.

Examples:
- "INFLATABLE POLITICAL GLOBE" -> Main_Category "Toys & Games", Subcategory "Toys & Dolls" (an inflatable toy, not decor)
- "HEARTS WRAPPING TAPE" -> "Stationery & Paper", "Pens, Pencils & Stickers" (tape is a stationery tool)
- "ZINC HEART LATTICE CHARGER" -> "Kitchen & Dining", "Plates & Bowls" (a charger is a decorative serving plate)

Answer for the product given, including its "StockCode"."""

In [ ]:
FIELDS = ["Main_Category", "Subcategory", "Confidence"]
DEFAULTS = {"Main_Category": "Other", "Subcategory": "Unclassified", "Confidence": "low"}

In [ ]:
SCHEMA = {
    "type": "object",
    "properties": {
        "results": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "StockCode": {"type": "string"},
                    "Main_Category": {"type": "string"},
                    "Subcategory": {"type": "string"},
                    "Confidence": {"type": "string", "enum": ["high", "medium", "low"]},
                },
                "required": ["StockCode"] + FIELDS,
            },
        }
    },
    "required": ["results"],
}

print(f"Prompt ready: {len(TAXONOMY)} main categories, "
      f"{sum(len(v['subcategories']) for v in TAXONOMY.values())} subcategories")

Prompt ready: 14 main categories, 54 subcategories


## Ollama classification | one product per request

A failed or unparsable request returns nothing and the item stays un-classified (re-run the loop cell to retry it).

In [ ]:
def parse_rows(content):
    data = json.loads(content)
    rows = data["results"] if isinstance(data, dict) else data
    out = {}
    for r in rows:
        if not isinstance(r, dict) or "StockCode" not in r:
            continue
        row = {}
        for f in FIELDS:
            v = r.get(f)
            row[f] = DEFAULTS[f] if v in (None, "") else str(v).strip()
        if row["Confidence"] not in ("high", "medium", "low"):
            row["Confidence"] = "low"
        if row["Main_Category"] not in TAXONOMY:
            row["Main_Category"], row["Confidence"] = "Other", "low"
        if row["Subcategory"] not in TAXONOMY[row["Main_Category"]]["subcategories"]:
            row["Subcategory"] = "Unclassified"
        out[str(r["StockCode"])] = row
    return out

In [ ]:
def request_items(items):
    try:
        resp = rq.post(
            OLLAMA_URL,
            json={
                "model": MODEL,
                "stream": False,
                "keep_alive": "30m",
                "format": SCHEMA,
                "options": {"temperature": 0, "num_ctx": 8192, "num_predict": 400},
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": json.dumps(items)},
                ],
            },
            timeout=600,
        )
        resp.raise_for_status()
        return parse_rows(resp.json()["message"]["content"])
    except (rq.RequestException, json.JSONDecodeError, KeyError, TypeError) as e:
        print(f"  !! request failed: {type(e).__name__}: {e}")
        return {}

In [ ]:
def classify_item(code, desc):
    parsed = request_items([{"StockCode": code, "Description": desc}])
    row = parsed.get(code)
    if row is None:
        return None
    return {"StockCode": code, **row}


{'StockCode': '10120',
 'Main_Category': 'Toys & Games',
 'Subcategory': 'Toys & Dolls',
 'Confidence': 'high'}

In [ ]:
first = to_classify.iloc[0]
classify_item(first["StockCode"], first["Description"])

## Classification loop with checkpointing

In [9]:
done = (
    pd.read_csv(CKPT_PATH, dtype={"StockCode": str})
    if os.path.exists(CKPT_PATH)
    else pd.DataFrame(columns=["StockCode"] + FIELDS)
)
done_codes = set(done["StockCode"])
todo = to_classify[~to_classify["StockCode"].isin(done_codes)]
items = todo.iloc[:TEST_ITEMS].to_dict("records") if TEST_MODE else todo.to_dict("records")
total = len(products)
print(f"{len(done)} already classified, {len(items)} to go in this run")

buffer, n_done, t0 = [], len(done), time.time()


def flush(buf):
    if buf:
        pd.DataFrame(buf).to_csv(
            CKPT_PATH, mode="a", header=not os.path.exists(CKPT_PATH), index=False
        )
        buf.clear()


with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {
        pool.submit(classify_item, it["StockCode"], it["Description"]): it for it in items
    }
    for fut in as_completed(futures):
        row = fut.result()
        if row is not None:
            buffer.append(row)
            if len(buffer) >= FLUSH_EVERY:
                flush(buffer)
        n_done += 1
        if n_done % 10 == 0:
            elapsed = time.time() - t0
            rate = (n_done - len(done)) / elapsed if elapsed > 0 else float("inf")
            eta_min = (len(done) + len(items) - n_done) / rate / 60 if rate > 0 else 0
            print(f"{n_done}/{len(done) + len(items)} classified | {rate:.0f} items/s | ETA {eta_min:.0f} min")

flush(buffer)
final = len(done) + len(items)
print(f"done: {final}/{total} products classified "
      f"({final / total * 100:.1f}% of catalog)")


30 already classified, 472 to go in this run
40/502 classified | 1 items/s | ETA 8 min
50/502 classified | 1 items/s | ETA 7 min
60/502 classified | 1 items/s | ETA 7 min
70/502 classified | 1 items/s | ETA 7 min
80/502 classified | 1 items/s | ETA 7 min
90/502 classified | 1 items/s | ETA 6 min
100/502 classified | 1 items/s | ETA 6 min
110/502 classified | 1 items/s | ETA 6 min
120/502 classified | 1 items/s | ETA 6 min
130/502 classified | 1 items/s | ETA 6 min
140/502 classified | 1 items/s | ETA 6 min
150/502 classified | 1 items/s | ETA 5 min
160/502 classified | 1 items/s | ETA 5 min
170/502 classified | 1 items/s | ETA 5 min
180/502 classified | 1 items/s | ETA 5 min
190/502 classified | 1 items/s | ETA 5 min
200/502 classified | 1 items/s | ETA 5 min
210/502 classified | 1 items/s | ETA 5 min
220/502 classified | 1 items/s | ETA 4 min
230/502 classified | 1 items/s | ETA 4 min
240/502 classified | 1 items/s | ETA 4 min
250/502 classified | 1 items/s | ETA 4 min
260/502 classif

## Build the product dimension

In [10]:
attrs = (
    pd.read_csv(CKPT_PATH, dtype={"StockCode": str}).drop_duplicates("StockCode")
    if os.path.exists(CKPT_PATH)
    else pd.DataFrame(columns=["StockCode"] + FIELDS)
)
product_dim = products.merge(attrs, on="StockCode", how="left", suffixes=("_pre", "_llm"))
for col in FIELDS:
    product_dim[col] = product_dim[f"{col}_pre"].where(
        product_dim[f"{col}_pre"].notna(), product_dim[f"{col}_llm"]
    )
product_dim = product_dim.drop(columns=[f"{c}_{s}" for c in FIELDS for s in ("pre", "llm")])

product_dim["Classification_Method"] = product_dim["Classification_Method"].fillna("llm")
product_dim["Main_Category"] = product_dim["Main_Category"].fillna("Unclassified")
product_dim["Subcategory"] = product_dim["Subcategory"].fillna("Unclassified")
product_dim["Confidence"] = product_dim["Confidence"].fillna("low")

review = product_dim[
    product_dim["Main_Category"].isin(["Other", "Unclassified"])
    | product_dim["Classification_Method"].eq("llm")
    | product_dim["Confidence"].eq("low")
]
product_dim.to_csv(PRODUCT_DIM_PATH, index=False)
review.to_csv(REVIEW_PATH, index=False)
print(f"{len(product_dim)} products -> {PRODUCT_DIM_PATH} | {len(review)} flagged for review")
product_dim.head(15)


4950 products -> ../data/processed/product_dimension.csv | 502 flagged for review


,StockCode,Description,Classification_Method,Main_Category,Subcategory,Confidence
0,10002,INFLATABLE POLITICAL GLOBE,rule:INFLATABLE,Toys & Games,Toys & Dolls,high
1,10002R,ROBOT PENCIL SHARPNER,rule:SHARPNER,Stationery & Paper,"Pens, Pencils & Stickers",high
2,10080,GROOVY CACTUS INFLATABLE,rule:INFLATABLE,Toys & Games,Toys & Dolls,high
3,10109,BENDY COLOUR PENCILS,rule:PENCIL,Stationery & Paper,"Pens, Pencils & Stickers",high
4,10120,DOGGY RUBBER,llm,Toys & Games,Toys & Dolls,high
5,10123C,HEARTS WRAPPING TAPE,rule:TAPE,Stationery & Paper,"Pens, Pencils & Stickers",high
6,10123G,ARMY CAMO WRAPPING TAPE,rule:TAPE,Stationery & Paper,"Pens, Pencils & Stickers",high
7,10124A,SPOTS ON RED BOOKCOVER TAPE,rule:TAPE,Stationery & Paper,"Pens, Pencils & Stickers",high
8,10124G,ARMY CAMO BOOKCOVER TAPE,rule:TAPE,Stationery & Paper,"Pens, Pencils & Stickers",high
9,10125,MINI FUNKY DESIGN TAPES,rule:TAPE,Stationery & Paper,"Pens, Pencils & Stickers",high


### Export category parquet 

In [ ]:
PARQUET_PATH = "../data/processed/product_categories.parquet"

product_categories = product_dim[["StockCode"]].copy()
product_categories["main_category"] = product_dim["Main_Category"].where(
    product_dim["Main_Category"].ne("Unclassified")
)
product_categories["sub_category"] = product_dim["Subcategory"].where(
    product_dim["Subcategory"].ne("Unclassified")
)
product_categories.to_parquet(PARQUET_PATH, index=False)

n_classified = product_categories["main_category"].notna().sum()
print(f"{len(product_categories)} rows -> {PARQUET_PATH}")
print(f"{n_classified} classified ({n_classified / len(product_categories) * 100:.1f}%), "
      f"{product_categories['main_category'].isna().sum()} left NULL for the SQL coalesce")
print(f"duplicated StockCode keys: {product_categories['StockCode'].duplicated().sum()} (must be 0)")
product_categories.head()

4950 rows -> ../data/processed/product_categories.parquet
4950 classified (100.0%), 0 left NULL for the SQL coalesce
duplicated StockCode keys: 0 (must be 0)


,StockCode,main_category,sub_category
0,10002,Toys & Games,Toys & Dolls
1,10002R,Stationery & Paper,"Pens, Pencils & Stickers"
2,10080,Toys & Games,Toys & Dolls
3,10109,Stationery & Paper,"Pens, Pencils & Stickers"
4,10120,Toys & Games,Toys & Dolls
